# Multi-day, SOC-aware EV charging sampler

This notebook runs the `multiday_charging` subproject end to end:

1. **Fleet** with different battery capacities and a preferred charging interval (charge every *k* days).
2. **Short-distance trip profile** with destination labels (Home / Work / Public), from the parent SDPTM table if present, else synthetic.
3. **Empirical session pool** (home, work, public L2, public DC) from `charging_session_all_clean.pkl`.
4. **Location probability matrix** (Home / Work / Public).
5. **Charging-frequency probability matrix** (charge every 1..7 days).
6. **SOC-aware sampling**: a vehicle charges when its preferred interval elapses **or** its battery would be depleted (so battery capacity lets a vehicle skip days).

It simulates **365 days** and plots (a) all 365 daily load curves and (b) the 5–95th-percentile band per hour.

> **All configuration lives in the single config cell below** — edit it to test different settings, then *Run All*.

In [7]:
## Bootstrap: make the subproject + parent `common.py` importable
import sys
import importlib
from pathlib import Path

_here = Path.cwd()
_mod_dir = None
for _c in (_here, _here / "multiday_charging", _here.parent):
    if (_c / "config.py").exists():
        _mod_dir = _c
        break
if _mod_dir is None:
    raise RuntimeError("Could not locate multiday_charging/config.py; run from the subproject folder.")

sys.path.insert(0, str(_mod_dir))            # multiday_charging/
sys.path.insert(0, str(_mod_dir.parent))     # Distribution-Grid-EV-CA/ (for common.py)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import config
import fleet as F
import trips as T
import session_pool as P
import choice as C
import sampling as S
import load_profile as L

# Pick up edits to .py files without restarting the kernel.
for _m in (config, F, T, P, C, S, L):
    importlib.reload(_m)

from config import MultiDayConfig

print("modules loaded from:", _mod_dir)
print("MultiDayConfig has store_sessions:", hasattr(MultiDayConfig, "__dataclass_fields__") and "store_sessions" in MultiDayConfig.__dataclass_fields__)

modules loaded from: e:\GitHub\good_model\Distribution-Grid-EV-CA\multiday_charging


## 1. Configuration (edit everything here)

This single cell holds the fleet mix, both probability matrices, and the daily-driving model. Change values and re-run the notebook to test scenarios.

In [9]:
cfg = MultiDayConfig(
    # ---- reproducibility / size ----
    seed=42,
    n_vehicles=25_000, #n_vehicles=2_500_000,
    n_days=365,
    # Full CA fleet: build hourly load only (no multi-hundred-GB session table).
    store_sessions=False,

    # ---- vehicle energy model ----
    # battery capacity mix {kWh: share}
    battery_mix={40.0: 0.15, 60.0: 0.35, 75.0: 0.25, 100.0: 0.20, 130.0: 0.05},
    efficiency_mi_per_kwh=3.0,   # 3 mi/kWh == legacy dist/3 rule
    reserve_soc=0.10,            # never deplete below 10% of pack

    # ---- (5) charging-FREQUENCY probability matrix: charge every k days ----
    charge_interval_probs={1: 0.45, 2: 0.25, 3: 0.15, 4: 0.07, 5: 0.04, 6: 0.02, 7: 0.02},

    # ---- (4) charging-LOCATION weights (Reference_Code RAW_SHARES) ----
    # Residential_L1_L2=0.68, Workplace_L2=0.04, Public_L2+DCFC=0.08+0.20=0.28
    location_weights={"home": 0.68, "work": 0.04, "public": 0.28},
    home_access_share=0.80, # share of vehicles that have home access
    work_access_share=0.30, # share of vehicles that have work access
    work_trip_prob=0.62, # probability of work trip
    public_trip_prob=0.85, # probability of public-eligible trip

    # ---- session sub-type weights (within public: DCFC vs Public_L2) ----
    public_level_weights={"DC": 0.20, "L2": 0.08},
    home_housing_weights={"single family": 56.4, "multi family": 38.9},

    # ---- daily driving model ----
    mean_daily_vmt=35.0,
    vmt_between_vehicle_cv=0.45, # between vehicle variability, SD/mean
    vmt_day_to_day_cv=0.55, # day-to-day variability, SD/mean
    no_travel_prob=0.08, # probability of no travel
)

# Optional: calibrate the fleet's daily VMT + work access from the real SDPTM
# trip table when it is available (set to False for a fully synthetic run).
CALIBRATE_FROM_REAL_TRIPS = True

rng = np.random.default_rng(cfg.seed)
cfg

TypeError: MultiDayConfig.__init__() got an unexpected keyword argument 'store_sessions'

## 2. Build the fleet (battery capacity + charging-frequency preference)

In [ ]:
fleet = F.build_fleet(cfg, rng)

# (2) short-distance trip profile with destination labels (real if available)
profiles = T.load_sd_trip_profiles(cfg)
if profiles is not None and CALIBRATE_FROM_REAL_TRIPS:
    print(f"Calibrating fleet from {len(profiles):,} real SDPTM vehicle profiles")
    fleet = T.calibrate_fleet_from_trips(cfg, fleet, profiles, rng)
else:
    print("Using synthetic daily-driving profile (no real SDPTM table found / calibration off)")

display(F.fleet_summary(fleet))
fleet.head()

Calibrating fleet from 2,220,087 real SDPTM vehicle profiles


,metric,value
0,n_vehicles,2500000.000
1,mean_battery_kwh,72.250
2,home_access_share,0.800
3,work_access_share,0.257
4,mean_interval_pref_days,2.140
5,mean_daily_vmt,79.370


,vehicle_id,battery_kwh,usable_kwh,eff_mi_per_kwh,home_access,work_access,interval_pref_days,mean_daily_vmt,_has_public_profile
0,0,100.0,90.0,3.0,False,True,1,58.400000,True
1,1,60.0,54.0,3.0,True,False,2,2.960000,True
2,2,100.0,90.0,3.0,True,False,2,80.680000,True
3,3,75.0,67.5,3.0,True,False,1,79.409999,True
4,4,40.0,36.0,3.0,True,False,3,45.280001,False


## 3. Daily driving inputs (365 days) and 4. empirical session pool

In [ ]:
# day-varying VMT + location availability over the 365-day horizon
daily = T.build_daily_inputs(cfg, fleet, rng)
print("daily VMT matrix:", daily["vmt"].shape,
      "| mean daily VMT:", round(float(daily["vmt"].mean()), 2), "mi")

# (3) empirical session pool (~6.5M cleaned sessions)
pool = P.build_session_pool(cfg, rng)
print("session pool source:", pool.attrs.get("source"), "| rows:", len(pool))
display(P.pool_coverage(pool).head(12))

daily VMT matrix: (2500000, 365) | mean daily VMT: 73.02 mi
session pool source: empirical | rows: 6515297


,charge_type,charger_level,bin,n
0,home,L2,1,407155
1,home,L2,2,69609
2,home,L2,3,29137
3,home,L2,4,13558
4,public,DC,1,473719
5,public,DC,2,1084208
6,public,DC,3,1129079
7,public,DC,4,797544
8,public,DC,5,415934
9,public,DC,6,201580


## 5–6. Run the SOC-aware multi-day sampler

In [ ]:
result = S.simulate(cfg, fleet, daily, pool, rng)
if isinstance(result, S.SimulateResult):
    display(result.summary)
    print("Hourly load matrix:", result.load.shape, "kW (fleet total per hour)")
else:
    display(S.sampling_summary(result, cfg))
    display(result.head())

MemoryError: Unable to allocate 23.1 MiB for an array with shape (1513110,) and data type complex128

## Build hourly load profiles (365 days x 24 hours)

In [ ]:
# total fleet load, and per charge-type
if isinstance(result, S.SimulateResult):
    load = result.load
    load_by_type = result.load_by_type
else:
    load = L.daily_hourly_matrix(result, cfg.n_days)          # [365, 24] kW
    load_by_type = L.hourly_load_by_type(result, cfg.n_days)  # dict type -> [365, 24]

# convert to MW per vehicle-scale view; here we keep kW (total simulated fleet)
band = L.percentile_band(load, lo=5, hi=95)
print("Overall peak hour-mean (kW):", round(load.mean(0).max(), 1))
display(band)

## Plot A — all 365 daily load curves

In [ ]:
out_dir = cfg.out_path()
hours = np.arange(24)

fig, ax = plt.subplots(figsize=(11, 6))
for d in range(cfg.n_days):
    ax.plot(hours, load[d], color="steelblue", alpha=0.05, lw=0.8)
ax.plot(hours, load.mean(0), color="black", lw=2.2, label="mean across days")
ax.set_xlabel("Hour of day")
ax.set_ylabel("Fleet charging load (kW)")
ax.set_title(f"Daily EV charging load — {cfg.n_days} days, {cfg.n_vehicles:,} vehicles")
ax.set_xticks(range(0, 25, 3))
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(out_dir / "daily_load_365_lines.png", dpi=200, bbox_inches="tight")
plt.show()
print("saved:", out_dir / "daily_load_365_lines.png")

## Plot B — 5–95th percentile band by hour

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
ax.fill_between(band["hour"], band["p_lo"], band["p_hi"],
                color="steelblue", alpha=0.30, label="5–95th percentile")
ax.plot(band["hour"], band["median"], color="navy", lw=2.2, label="median")
ax.plot(band["hour"], band["mean"], color="darkorange", lw=1.8, ls="--", label="mean")
ax.set_xlabel("Hour of day")
ax.set_ylabel("Fleet charging load (kW)")
ax.set_title("Hourly charging load — daily variability (5–95th percentile)")
ax.set_xticks(range(0, 25, 3))
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(out_dir / "hourly_load_percentile_band.png", dpi=200, bbox_inches="tight")
plt.show()
print("saved:", out_dir / "hourly_load_percentile_band.png")

## Plot C — mean daily load decomposed by charge type

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
order = [t for t in ("home", "work", "public") if t in load_by_type]
means = [load_by_type[t].mean(0) for t in order]
ax.stackplot(hours, *means, labels=order, alpha=0.85)
ax.set_xlabel("Hour of day")
ax.set_ylabel("Mean fleet charging load (kW)")
ax.set_title("Mean daily charging load by location type")
ax.set_xticks(range(0, 25, 3))
ax.legend(loc="upper left")
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(out_dir / "mean_load_by_type.png", dpi=200, bbox_inches="tight")
plt.show()

# persist the matrices for downstream use
np.save(out_dir / "daily_hourly_load_kW.npy", load)
band.to_csv(out_dir / "hourly_percentile_band.csv", index=False)
if not isinstance(result, S.SimulateResult):
    result.to_parquet(out_dir / "sampled_sessions.parquet")
print("outputs written to:", out_dir)

## Notes & extensions

- **Battery capacity** enters through `usable_kwh = battery * (1 - reserve_soc)`: a vehicle is forced to charge once cumulative depletion reaches the usable pack, so larger packs skip more days even if the frequency preference is short.
- **Charging-frequency matrix** (`charge_interval_probs`) sets each vehicle's *preferred* interval; battery depletion can override it.
- **Location matrix** (`location_weights`) plus per-day feasibility (home access, work visit, public availability) decides where each charge happens.
- To produce **block / TAZ-level** curves, attach a `zone` column to `sessions` (e.g. from the block→TAZ crosswalk) and call `L.grouped_daily_hourly(sessions, "zone", cfg.n_days)`.
- The empirical pool at `data/charging data/charging_session_all_clean.pkl` is required (rebuild with `build_multiday_inputs.py` if needed).